[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juliopez/Taller-Fundamentos-Data-Science-Python/blob/main/Machine_Learning_2026/03_Notebooks/NB08_Aplicacion_Despliegue_Modelo.ipynb)

# NB08 — Aplicación y despliegue del modelo CNN

**Último notebook del semestre · continuidad NB04 → NB07 → NB08.**

**Objetivo:** construir, probar y desplegar una aplicación web real con **Streamlit** que clasifique imágenes de **piedra, papel o tijera**, usando la CNN del NB04 exportada a ONNX en NB07.

**Producto final:** un repositorio GitHub con `app.py`, modelo y metadatos; una aplicación en **Streamlit Community Cloud** con URL pública funcional; y pruebas con imágenes nuevas.

**Contrato que no debemos cambiar:** entrada **RGB, 128 × 128, `float32 / 255.0`**, salida en orden **`paper`, `rock`, `scissors`**. No utilizamos MobileNetV2 ni `preprocess_input`.

**Ruta de trabajo:** recuperar archivos desde Drive → verificar contrato → probar ONNX → generar aplicación → descargar proyecto → subir a GitHub → desplegar → probar la URL. **Colab prepara el proyecto; no publica automáticamente en GitHub ni en Streamlit Cloud.**

## Mapa de aprendizaje

**NB04 → NB07 → NB08:** entrenar una CNN → exportarla a ONNX → crear una aplicación web y desplegarla. En NB08 **no volvemos a entrenar**: reutilizamos los artefactos verificados del NB07.

**Producto esperado:** URL funcional, repositorio con seis archivos y registro de pruebas. Las tres figuras de este notebook se generan con Python en Colab; no dependen de imágenes externas.

In [ ]:
# Figura 1: continuidad de los notebooks
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
fig, ax = plt.subplots(figsize=(11, 2.5)); ax.set_xlim(0, 11); ax.set_ylim(0, 2.5); ax.axis("off")
for x, titulo, subtitulo in [(0.2,"NB04","Entrenar CNN"),(3.95,"NB07","Exportar ONNX"),(7.7,"NB08","Desplegar web")]:
    ax.add_patch(FancyBboxPatch((x,.5),3.0,1.5,boxstyle="round,pad=.12",facecolor="#e8f0fe",edgecolor="#3868a8"))
    ax.text(x+1.5,1.45,titulo,ha="center",weight="bold",fontsize=14)
    ax.text(x+1.5,.95,subtitulo,ha="center",fontsize=11)
for x in (3.25,7):
    ax.annotate("",xy=(x+.65,1.25),xytext=(x,1.25),arrowprops=dict(arrowstyle="->",lw=2))
plt.show()

## 1. Preparación del entorno

Instalaremos **Streamlit** (interfaz web) y **ONNX Runtime** (motor de inferencia). `numpy`, `pandas` y `Pillow` ayudan a preparar imágenes y mostrar resultados.

**Resultado esperado:** confirmación de que el entorno está preparado. La instalación en Colab es temporal; `requirements.txt` permitirá repetirla en Streamlit Community Cloud.

In [ ]:
!pip -q install streamlit onnxruntime

import os
import json
import shutil
from pathlib import Path

print("Entorno preparado")


## 2. Conectar Google Drive

Autoriza el acceso a **Mi unidad**. Recuperaremos los archivos generados en NB07 desde `Mi unidad/MachineLearning2026/implementacion/`.

**Importante:** conectar Drive no despliega ni publica archivos. Solo permite leer los artefactos que ya generaste.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Recuperar los artefactos generados en NB07

NB07 guardó los siguientes archivos en `Mi unidad/MachineLearning2026/implementacion/`:

- **`modelo_cnn.onnx`**: la CNN original del NB04 exportada a ONNX.
- **`class_names.json`**: el orden de las categorías (`paper`, `rock`, `scissors`).
- **`model_contract.json`**: tamaño de entrada, tipo de datos y normalización.

**No utilizaremos `modelo_quantized.tflite`**: ese archivo corresponde al experimento de cuantificación del NB07, mientras que aquí desplegamos el ONNX original.

**Resultado esperado:** los tres archivos aparecen como disponibles. Si falta alguno, vuelve al NB07 y ejecuta las celdas de guardado; no inventes un modelo nuevo.

In [ ]:
SOURCE_DIR = Path("/content/drive/MyDrive/MachineLearning2026/implementacion")
NOMBRES = ("modelo_cnn.onnx", "class_names.json", "model_contract.json")
for nombre in NOMBRES:
    ruta = SOURCE_DIR / nombre
    print(("OK" if ruta.is_file() else "FALTA"), ruta)
faltantes = [nombre for nombre in NOMBRES if not (SOURCE_DIR / nombre).is_file()]
assert not faltantes, f"Faltan archivos de NB07: {faltantes}. Completa NB07 antes de continuar."
MODEL_SOURCE = SOURCE_DIR / "modelo_cnn.onnx"
CLASSES_SOURCE = SOURCE_DIR / "class_names.json"
CONTRACT_SOURCE = SOURCE_DIR / "model_contract.json"

## 4. Crear la estructura del proyecto

Crearemos una carpeta de trabajo **temporal en Colab**: `/content/ml-image-classifier/`. Copiaremos el ONNX y sus metadatos sin alterar los originales en Drive.

```text
ml-image-classifier/
├── app.py
├── modelo_cnn.onnx
├── class_names.json
├── model_contract.json
├── requirements.txt
└── README.md
```

Al finalizar podrás **descargar todo como ZIP** y subir los archivos a un repositorio GitHub. **No subas el ZIP sin extraerlo**: Streamlit Cloud necesita encontrar `app.py` y `requirements.txt` en el repositorio.

In [ ]:
PROJECT_DIR = Path("/content/ml-image-classifier")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
for nombre in NOMBRES:
    shutil.copy2(SOURCE_DIR / nombre, PROJECT_DIR / nombre)
print("Proyecto creado:", PROJECT_DIR)
print("Artefactos copiados:", *NOMBRES, sep="\n- ")

## 5. Verificar clases y contrato del modelo

**¿Por qué importa?** El archivo ONNX contiene operaciones matemáticas, pero la aplicación necesita saber **qué clase representa cada salida** y **cómo preparar la imagen**.

Verificaremos las reglas del NB04/NB07: tres categorías en el orden correcto; RGB; entrada de 128 × 128; píxeles divididos por 255. **Si alguna regla no coincide, detendremos el notebook antes de crear la aplicación.**

### Analogía: el contrato es un idioma compartido

El modelo aprendió a recibir imágenes en un **idioma numérico**: RGB, 128 × 128, `float32` y valores entre 0 y 1. La aplicación debe hablar ese mismo idioma. `class_names.json` es el **diccionario** que traduce índices de salida a nombres; `model_contract.json` documenta las reglas. Cambiar la entrada sin adaptar el modelo puede provocar errores o predicciones poco fiables.

In [ ]:
with open(PROJECT_DIR / "class_names.json", encoding="utf-8") as archivo:
    CLASS_NAMES = json.load(archivo)
with open(PROJECT_DIR / "model_contract.json", encoding="utf-8") as archivo:
    CONTRACT = json.load(archivo)
assert CLASS_NAMES == ["paper", "rock", "scissors"], f"Orden inesperado: {CLASS_NAMES}"
assert CONTRACT["clases"] == CLASS_NAMES, "El contrato y las clases no coinciden"
assert CONTRACT["input_shape"] == [1, 128, 128, 3], "Se espera entrada de 128 × 128 RGB"
assert CONTRACT["tipo_entrada"] == "float32", "Se espera float32"
assert "255" in CONTRACT["preprocesamiento"], "Falta la normalización /255"
print("Contrato verificado:", CONTRACT)
for indice, clase in enumerate(CLASS_NAMES):
    print(indice, "→", clase)

## 6. Probar ONNX Runtime antes de crear la interfaz

ONNX Runtime permite ejecutar el modelo **sin volver a entrenarlo**. Inspeccionaremos los nombres, tipos y dimensiones de sus tensores.

**Resultado esperado:** entrada compatible con `(lote, 128, 128, 3)` y salida con **tres probabilidades**. Un archivo válido pero incompatible con el contrato **no debe desplegarse**.

In [ ]:
import onnxruntime as ort
session = ort.InferenceSession(str(PROJECT_DIR / "modelo_cnn.onnx"),
                               providers=["CPUExecutionProvider"])
input_info, output_info = session.get_inputs()[0], session.get_outputs()[0]
assert input_info.type == "tensor(float)", f"Tipo de entrada inesperado: {input_info.type}"
assert list(input_info.shape[1:]) == [128, 128, 3], f"Forma incompatible: {input_info.shape}"
assert output_info.shape[-1] == len(CLASS_NAMES), "La salida debe tener tres probabilidades"
print("Entrada:", input_info.name, input_info.shape, input_info.type)
print("Salida:", output_info.name, output_info.shape, output_info.type)

## 7. Preparar la imagen exactamente como en NB04

El **pipeline de inferencia** es la secuencia de transformaciones desde la imagen hasta las probabilidades:

**Imagen → RGB → redimensionar a 128 × 128 → `float32` → dividir entre 255 → ONNX → clase y confianza.**

**Analogía:** si entrenamos a alguien para reconocer objetos observando fotografías con una escala de brillo determinada, cambiar esa escala al evaluarlo altera lo que percibe. En una CNN, omitir la división entre 255 puede afectar gravemente las predicciones.

**Importante:** la **confianza** es la probabilidad softmax más alta; **no equivale** a una garantía de acierto. Usaremos estas mismas reglas en Colab y en `app.py`.

**Comprobación:** redimensionar a 128 × 128 no basta. También deben coincidir canales RGB, tipo `float32` y división por 255. No usamos el preprocesamiento de MobileNetV2.

In [ ]:
import numpy as np
from PIL import Image
IMG_SIZE = (128, 128)

def preparar_imagen(imagen):
    imagen = imagen.convert("RGB").resize(IMG_SIZE)
    entrada = np.asarray(imagen, dtype=np.float32)[None, ...] / 255.0
    assert entrada.shape == (1, 128, 128, 3)
    return entrada

def predecir(imagen):
    entrada = preparar_imagen(imagen)
    probabilidades = session.run([output_info.name],
                                 {input_info.name: entrada})[0][0]
    indice = int(np.argmax(probabilidades))
    confianza = float(probabilidades[indice])
    return CLASS_NAMES[indice], confianza, probabilidades

print("Pipeline listo: RGB → 128 × 128 → float32 / 255 → ONNX")

## 8. Probar una imagen real sin depender de rutas manuales

Descargaremos **una imagen de prueba** desde los **mismos ZIP de piedra, papel y tijera utilizados en NB04** y NB07, alojados en el repositorio GitHub docente. Así evitamos depender de un archivo `imagen_prueba.jpg` que podría no existir en tu Drive.

**Resultado esperado:** se muestra la imagen, su clase real, la predicción ONNX y la confianza. Esta prueba comprueba el flujo técnico; **una sola imagen no mide la exactitud general**.

In [ ]:
from urllib.request import urlretrieve
from zipfile import ZipFile
from IPython.display import display

TEST_DIR = Path("/content/dataset_cnn/test")
if not all(list((TEST_DIR / clase).glob("*.png")) for clase in CLASS_NAMES):
    base_url = ("https://raw.githubusercontent.com/juliopez/"
                "Taller-Fundamentos-Data-Science-Python/main/"
                "Machine_Learning_2026/02_Datasets/dataset_cnn/")
    for clase in CLASS_NAMES:
        nombre = f"dataset_cnn_{clase}.zip"
        destino = Path("/content") / nombre
        urlretrieve(base_url + nombre, destino)
        with ZipFile(destino) as archivo:
            for miembro in archivo.infolist():
                partes = Path(miembro.filename)
                if partes.is_absolute() or ".." in partes.parts or not partes.parts or partes.parts[0] != "dataset_cnn":
                    raise ValueError(f"Ruta insegura en ZIP: {miembro.filename}")
            archivo.extractall("/content")

IMAGE_PATH = sorted((TEST_DIR / "rock").glob("*.png"))[0]
with Image.open(IMAGE_PATH) as archivo:
    imagen = archivo.convert("RGB")
display(imagen)
clase, confianza, probabilidades = predecir(imagen)
print("Clase real:", IMAGE_PATH.parent.name)
print("Predicción:", clase)
print(f"Confianza: {confianza:.2%}")

## 9. Interpretar probabilidades por clase

El modelo entrega **tres valores**, uno por categoría. La clase con el valor más alto es la predicción; los otros valores también importan porque muestran **qué alternativas considera plausibles**.

**Atención:** una confianza alta no demuestra que el modelo haya acertado. Para medir calidad predictiva necesitamos imágenes etiquetadas y métricas como exactitud o matriz de confusión.

### ¿Qué significa la barra más alta?

Es la clase que el modelo considera más probable, **no una garantía de acierto**. Compara la predicción con la clase real de la imagen. Si dos probabilidades son similares, el modelo está menos decidido entre esas alternativas.

In [ ]:
# Figura 2: imagen real y probabilidades calculadas por ONNX
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(imagen); ax1.axis("off"); ax1.set_title(f"Clase real: {IMAGE_PATH.parent.name}")
bars = ax2.bar(CLASS_NAMES, probabilidades, color=["#4677bb","#e6a23c","#63a779"])
ax2.set_ylim(0,1.08); ax2.set_ylabel("Probabilidad"); ax2.set_title(f"Predicción: {clase}")
for b,p in zip(bars,probabilidades):
    ax2.text(b.get_x()+b.get_width()/2,float(p)+.02,f"{p:.1%}",ha="center")
plt.tight_layout(); plt.show()
print("¿Predicción correcta en esta imagen?", clase == IMAGE_PATH.parent.name)

In [ ]:
import pandas as pd

resultado = pd.DataFrame({
    "Clase": CLASS_NAMES,
    "Probabilidad": probabilidades
}).sort_values(
    "Probabilidad",
    ascending=False
)

resultado


## 10. Construir la aplicación web con Streamlit

Streamlit permite crear una interfaz web desde Python sin desarrollar HTML ni JavaScript manualmente.

La aplicación permitirá **cargar una imagen PNG/JPG**, visualizarla, pulsar **Clasificar** y consultar la **clase predicha, confianza y probabilidades de las tres clases**. También mostrará una advertencia orientativa cuando la confianza sea menor al **60 %**.

**¿Qué hace el código?** Escribe `app.py` en la carpeta del proyecto. **No inicia todavía un servicio público**. El archivo reproducirá el mismo contrato del NB04/NB07 y leerá el modelo y los metadatos desde su propia carpeta.

### Tres piezas de `app.py`

**Interfaz:** `st.file_uploader` recibe la imagen, `st.image` la muestra, `st.button` solicita la clasificación y `st.dataframe` presenta las probabilidades.

**Preparación:** `predecir` convierte a RGB, 128 × 128 y `float32 / 255`.

**Motor:** `onnxruntime.InferenceSession` carga el modelo y `session.run` calcula la salida. `@st.cache_resource` evita recargar el modelo en cada interacción; `@st.cache_data` reutiliza metadatos.

**Pregunta:** ¿dónde cambiarías el título sin alterar el modelo? En la **interfaz**.

In [ ]:
app_code = r'''
import json
from pathlib import Path
import numpy as np
import pandas as pd
import streamlit as st
import onnxruntime as ort
from PIL import Image, UnidentifiedImageError

BASE_DIR = Path(__file__).resolve().parent
st.set_page_config(page_title="Piedra, papel o tijera", page_icon="✋")
st.title("Piedra, papel o tijera")
st.write("Clasifica una imagen con la CNN del NB04, exportada a ONNX en NB07.")

@st.cache_resource
def cargar_modelo():
    return ort.InferenceSession(str(BASE_DIR / "modelo_cnn.onnx"),
                                providers=["CPUExecutionProvider"])

@st.cache_data
def cargar_metadatos():
    with open(BASE_DIR / "class_names.json", encoding="utf-8") as archivo:
        clases = json.load(archivo)
    with open(BASE_DIR / "model_contract.json", encoding="utf-8") as archivo:
        contrato = json.load(archivo)
    if clases != ["paper", "rock", "scissors"] or contrato["clases"] != clases:
        raise ValueError("El orden de clases no coincide con el NB04.")
    if contrato["input_shape"] != [1, 128, 128, 3]:
        raise ValueError("El tamaño de entrada debe ser 128 × 128 RGB.")
    return clases, contrato

try:
    session = cargar_modelo()
    class_names, contrato = cargar_metadatos()
    entrada_info = session.get_inputs()[0]
    salida_info = session.get_outputs()[0]
    if list(entrada_info.shape[1:]) != [128, 128, 3] or salida_info.shape[-1] != 3:
        raise ValueError("La forma del modelo ONNX es incompatible con el contrato.")
except Exception as exc:
    st.error(f"No se pudo cargar el modelo o sus metadatos: {exc}")
    st.stop()

def predecir(imagen):
    imagen = imagen.convert("RGB").resize((128, 128))
    x = np.asarray(imagen, dtype=np.float32)[None, ...] / 255.0
    probs = session.run([salida_info.name], {entrada_info.name: x})[0][0]
    indice = int(np.argmax(probs))
    return indice, float(probs[indice]), probs

archivo = st.file_uploader("Selecciona una imagen PNG o JPG",
                           type=["png", "jpg", "jpeg"])
if archivo is not None:
    try:
        imagen = Image.open(archivo)
        imagen.load()
        st.image(imagen, caption="Imagen seleccionada", use_container_width=True)
    except (UnidentifiedImageError, OSError, ValueError):
        st.error("El archivo no es una imagen válida.")
        st.stop()

    if st.button("Clasificar", type="primary"):
        indice, confianza, probabilidades = predecir(imagen)
        st.subheader("Resultado")
        st.write(f"**Clase predicha:** {class_names[indice]}")
        st.write(f"**Confianza:** {confianza:.2%}")
        if confianza < 0.60:
            st.warning("Confianza inferior al 60 %: interpreta el resultado con cautela.")
        tabla = pd.DataFrame({"Clase": class_names,
                              "Probabilidad": probabilidades})
        st.subheader("Probabilidades por clase")
        st.dataframe(tabla.sort_values("Probabilidad", ascending=False),
                     hide_index=True, use_container_width=True)
        st.bar_chart(tabla.set_index("Clase")["Probabilidad"])
        st.caption("La confianza softmax no garantiza que la predicción sea correcta.")
'''
with open(PROJECT_DIR / "app.py", "w", encoding="utf-8") as archivo:
    archivo.write(app_code)
print("Aplicación creada:", PROJECT_DIR / "app.py")

## 11. Crear `requirements.txt`

**¿Para qué sirve?** Cuando Streamlit Community Cloud recibe el repositorio GitHub, necesita saber qué bibliotecas instalar. Este archivo declara las dependencias de la aplicación.

Usaremos únicamente las bibliotecas necesarias para **ejecutar** el modelo, no para volver a entrenarlo.

`requirements.txt` es la **lista de ingredientes** del entorno: Streamlit Cloud la utiliza para instalar bibliotecas. Un código que funciona en Colab puede fallar en la nube si faltan dependencias.

In [ ]:
requirements = """streamlit
onnxruntime
numpy
pandas
Pillow
"""
(PROJECT_DIR / "requirements.txt").write_text(requirements, encoding="utf-8")
print(requirements)

## 12. Documentar el proyecto con `README.md`

El README explicará el propósito de la aplicación, su origen **NB04 → NB07 → NB08**, los archivos y la ejecución local. Una buena documentación permite que otra persona entienda qué hace el producto sin abrir todos los notebooks.

### Función de cada archivo

| Archivo | Responsabilidad |
|---|---|
| `app.py` | Interfaz y ejecución de inferencia |
| `modelo_cnn.onnx` | Modelo entrenado |
| `class_names.json` | Orden y nombre de las clases |
| `model_contract.json` | Reglas de entrada |
| `requirements.txt` | Dependencias |
| `README.md` | Instrucciones para otras personas |

El código, el modelo y sus metadatos **son componentes diferentes** y deben conservarse juntos.

In [ ]:
readme = """# Clasificador de piedra, papel o tijera

Aplicación web con **Streamlit** y **ONNX Runtime** que clasifica imágenes
en `paper`, `rock` o `scissors`.

## Origen del modelo

- **NB04:** entrenamiento de la CNN.
- **NB07:** exportación del modelo original a ONNX.
- **NB08:** aplicación y despliegue.

## Contrato de inferencia

Imagen RGB, tamaño 128 × 128, tensor float32 normalizado dividiendo por 255.
Orden de las salidas: `paper`, `rock`, `scissors`.

## Archivos

- `app.py`: interfaz e inferencia.
- `modelo_cnn.onnx`: CNN exportada desde NB07.
- `class_names.json`: categorías y su orden.
- `model_contract.json`: contrato de entrada.
- `requirements.txt`: bibliotecas de ejecución.

## Ejecutar localmente

```bash
pip install -r requirements.txt
streamlit run app.py
```

## Despliegue

Subir estos archivos a un repositorio GitHub y seleccionar `app.py`
como archivo principal en Streamlit Community Cloud.

**Nota:** la confianza softmax no garantiza exactitud.
"""
(PROJECT_DIR / "README.md").write_text(readme, encoding="utf-8")
print("README creado:", PROJECT_DIR / "README.md")

## 13. Revisar los archivos generados

Comprueba que estén presentes **seis archivos**, incluyendo `model_contract.json`. La carpeta `/content/ml-image-classifier` es temporal: si reinicias Colab, deberás recrearla ejecutando las celdas anteriores.

In [ ]:
for archivo in PROJECT_DIR.iterdir():
    tamaño = archivo.stat().st_size / (1024 ** 2)

    print(
        f"{archivo.name:25} {tamaño:.2f} MB"
    )


## 14. Validar el proyecto antes de publicarlo

**No basta con que exista `app.py`.** Verificaremos que todos los archivos requeridos estén presentes, que el código Python pueda compilarse y que la aplicación mantenga el preprocesamiento del NB04.

Si alguna comprobación falla, **corrígela antes de subir el proyecto a GitHub**.

In [ ]:
import ast
archivos_requeridos = ["app.py", "modelo_cnn.onnx", "class_names.json",
                       "model_contract.json", "requirements.txt", "README.md"]
faltantes = [nombre for nombre in archivos_requeridos
             if not (PROJECT_DIR / nombre).is_file()]
assert not faltantes, f"Faltan archivos: {faltantes}"
ast.parse((PROJECT_DIR / "app.py").read_text(encoding="utf-8"))
assert " / 255.0" in app_code and "(128, 128)" in app_code, "Preprocesamiento incorrecto"
print("OK: seis archivos presentes, Python válido y preprocesamiento coherente.")

## 15. Probar el arranque de Streamlit en Google Colab

Esta celda inicia Streamlit **solo como prueba técnica dentro de Colab** y revisa si responde en el puerto 8501. **No genera una URL pública permanente** ni sustituye el despliegue de la sección 18.

**Resultado esperado:** el servidor responde correctamente. Si el proceso falla, revisa `/content/streamlit.log` antes de continuar.

**Diferencia clave:** que Streamlit responda en el puerto de Colab solo demuestra que el servidor local inicia. **No significa** que exista una URL pública permanente ni que la aplicación desplegada haya sido validada.

In [ ]:
import subprocess
import time
import urllib.request

proceso_streamlit = subprocess.Popen(
    ["streamlit", "run", str(PROJECT_DIR / "app.py"),
     "--server.headless=true", "--server.port=8501"],
    stdout=open("/content/streamlit.log", "w"),
    stderr=subprocess.STDOUT
)
respuesta = None
for _ in range(15):
    if proceso_streamlit.poll() is not None:
        break
    try:
        with urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=2) as r:
            respuesta = r.status
        if respuesta == 200:
            break
    except Exception:
        time.sleep(1)
print("Salud del servidor:", respuesta)
if respuesta != 200:
    print(Path("/content/streamlit.log").read_text(encoding="utf-8")[-3000:])
    raise RuntimeError("Streamlit no inició correctamente")
print("OK: Streamlit responde localmente. Aún falta desplegarlo en la nube.")

## 16. Descargar el proyecto completo para GitHub

**Paso esencial:** descarga el archivo ZIP que crearemos con los seis archivos del proyecto. Después **descomprímelo** en tu computador.

En GitHub deberás subir los archivos extraídos **a la raíz de un repositorio** (no el ZIP). El repositorio será la fuente del despliegue en Streamlit Community Cloud.

**Importante:** el archivo ONNX es parte del proyecto; no basta con subir únicamente `app.py`.

In [ ]:
from shutil import make_archive
from google.colab import files

ZIP_PATH = make_archive("/content/ml-image-classifier", "zip", root_dir=PROJECT_DIR)
print("Proyecto comprimido:", ZIP_PATH)
files.download(ZIP_PATH)
print("Extrae el ZIP antes de subir sus seis archivos a GitHub.")


## 17. Crear el repositorio en GitHub

1. Abre [GitHub](https://github.com/) y crea un repositorio, por ejemplo `ml-image-classifier`.
2. Descarga y **extrae** el ZIP generado en Colab.
3. En el repositorio, usa **Add file → Upload files** para subir **los seis archivos extraídos**.
4. Confirma los cambios y verifica que en la raíz estén `app.py`, `modelo_cnn.onnx`, `class_names.json`, `model_contract.json`, `requirements.txt` y `README.md`.
5. Comprueba que GitHub haya aceptado el archivo ONNX. Si supera los límites de carga, **no omitas el modelo**: solicita una alternativa de distribución antes de desplegar.

**Resultado esperado:** un repositorio GitHub con los seis archivos y un README legible. La publicación del repositorio es una acción manual del estudiante.

## 18. Desplegar en Streamlit Community Cloud

1. Entra a [Streamlit Community Cloud](https://share.streamlit.io/) e inicia sesión con GitHub.
2. Selecciona **Create app** y elige tu repositorio.
3. Selecciona la rama que contiene los seis archivos (habitualmente `main`).
4. Especifica **`app.py`** como archivo principal.
5. Inicia el despliegue y espera a que se instalen las dependencias de `requirements.txt`.
6. Abre la **URL pública** generada por Streamlit y prueba una imagen de piedra, una de papel y una de tijera.

**¿Qué sucede detrás de escena?** Streamlit descarga el repositorio, instala dependencias, inicia `app.py` y utiliza ONNX Runtime para inferir. No vuelve a entrenar la CNN.

**Si falla:** revisa los registros (*logs*) de Streamlit Cloud. Comprueba primero los archivos, las dependencias, el contrato y la ruta del modelo.

**Resultado esperado:** una **URL pública funcional**, no solo una captura del código en Colab.

### Colab, GitHub y Streamlit Cloud no son equivalentes

**Colab = taller:** prepara y verifica archivos. **GitHub = repositorio:** guarda y versiona el producto. **Streamlit Cloud = servidor:** instala dependencias y ejecuta la aplicación accesible por URL. La URL final **no apunta a Colab**.

## 19. Flujo completo del despliegue

```text
NB04: entrenar CNN de piedra, papel o tijera
                    ↓
NB07: exportar ONNX y guardar metadatos en Drive
                    ↓
NB08 en Colab: verificar → crear app.py y requirements.txt
                    ↓
Descargar ZIP → extraer → subir seis archivos a GitHub
                    ↓
Streamlit Community Cloud → desplegar app.py
                    ↓
URL pública → usuario sube imagen
                    ↓
RGB → 128 × 128 → float32 / 255 → ONNX Runtime
                    ↓
Clase (paper/rock/scissors) + confianza + probabilidades
```

**Colab no es el servidor final:** la aplicación pública se ejecuta en Streamlit Community Cloud.

In [ ]:
# Figura 3: responsabilidades de cada plataforma
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
fig, ax = plt.subplots(figsize=(11, 3)); ax.set_xlim(0,11); ax.set_ylim(0,3); ax.axis("off")
for x,t,d,c in [(0.2,"Colab","Preparar y probar","#fff1d6"),(3.95,"GitHub","Guardar versiones","#e4e9ef"),(7.7,"Streamlit Cloud","Ejecutar y publicar","#e0f4eb")]:
    ax.add_patch(FancyBboxPatch((x,.8),3,1.5,boxstyle="round,pad=.12",facecolor=c,edgecolor="#59636e"))
    ax.text(x+1.5,1.72,t,ha="center",weight="bold",fontsize=12)
    ax.text(x+1.5,1.23,d,ha="center",fontsize=10)
for x in (3.25,7):
    ax.annotate("",xy=(x+.65,1.5),xytext=(x,1.5),arrowprops=dict(arrowstyle="->",lw=2))
ax.text(5.5,.25,"La URL pública corresponde a Streamlit Cloud, no a Colab",ha="center")
plt.show()

## 20. Pruebas funcionales después del despliegue

Una aplicación que abre su página no está necesariamente validada. Prueba al menos:

1. **Piedra (`rock`)**: imagen nueva de esa clase.
2. **Papel (`paper`)**: imagen nueva de esa clase.
3. **Tijera (`scissors`)**: imagen nueva de esa clase.
4. **Imagen difícil o ambigua**: observar probabilidades y confianza.
5. **Archivo no válido**: verificar que la interfaz lo rechace o muestre un mensaje comprensible.
6. **Imagen diferente de las utilizadas en entrenamiento**: comprobar que el flujo admite entradas nuevas.

**Anota los resultados reales.** No inventes una exactitud general a partir de estas pruebas funcionales.

### Ejemplo de prueba documentada

**Prueba:** cargar una fotografía nueva de tijera. **Esperado:** la aplicación acepta el archivo, muestra la imagen y entrega tres probabilidades. **Observado:** debe completarse con el resultado real, incluso si predice `rock` por error.

Una **prueba funcional** verifica que el flujo opera; una **evaluación predictiva** mide aciertos y errores sobre datos etiquetados. Una aplicación técnicamente correcta puede clasificar mal.

## 21. Registrar las pruebas realizadas

Completa la tabla **después de probar la URL pública**. Anota qué imagen utilizaste, qué esperabas y qué ocurrió realmente. Si una predicción es incorrecta, regístrala: el objetivo es verificar el comportamiento del producto, no ocultar errores.

In [ ]:
pruebas = pd.DataFrame({
    "Prueba": ["Piedra (rock)", "Papel (paper)", "Tijera (scissors)",
               "Imagen difícil", "Archivo inválido", "Imagen nueva"],
    "Resultado esperado": ["", "", "", "", "", ""],
    "Resultado observado": ["", "", "", "", "", ""],
    "Estado (OK/Falla)": ["", "", "", "", "", ""]
})
display(pruebas)

## 22. Versionado del producto

Un modelo y una aplicación son componentes diferentes. Puedes actualizar la interfaz sin reentrenar la CNN, o sustituir el modelo sin cambiar demasiado la interfaz.

**Ejemplo de registro:**

- **Modelo v1.0:** CNN original del NB04, exportada a ONNX en NB07.
- **Aplicación v1.0:** interfaz Streamlit desplegada desde GitHub.

**No llames «modelo optimizado» al ONNX solo por su formato:** en NB07 exportamos la CNN original; cuantificación y poda fueron experimentos independientes. Registra los cambios reales mediante los commits de GitHub.

## 23. Integración continua: de GitHub a la aplicación

Cuando actualices archivos del repositorio, Streamlit Community Cloud puede reconstruir o actualizar la aplicación.

```text
Cambio probado en app.py o artefactos
            ↓
Commit en GitHub
            ↓
Actualización / reconstrucción en Streamlit Cloud
            ↓
Verificación de la URL y de las predicciones
```

**Antes de reemplazar el modelo:** comprueba entrada 128 × 128 × 3, normalización, orden de clases, salida de tres probabilidades y funcionamiento de la interfaz. **Un cambio en el modelo puede exigir actualizar también los metadatos y el código.**

## 24. Criterios para aceptar una versión desplegada

Distingue **pruebas funcionales** de **métricas de desempeño**.

**Criterios funcionales mínimos:**
- La URL pública abre sin errores.
- El modelo ONNX carga correctamente.
- Se pueden subir imágenes PNG/JPG.
- Se muestra clase, confianza y probabilidades.
- El preprocesamiento es RGB, 128 × 128 y `float32 / 255`.
- El orden de clases es `paper`, `rock`, `scissors`.
- Las pruebas de las tres clases se registran con resultados reales.

**Métricas predictivas:** exactitud, recall por clase y latencia deben medirse sobre datos y condiciones definidas. No establecemos arbitrariamente `accuracy ≥ 85 %` ni `recall ≥ 70 %` sin evidencia del dataset y del objetivo del proyecto.

**Ejemplo de aceptación:** si la aplicación carga imágenes y muestra probabilidades, supera una prueba funcional básica. Si falla al clasificar varias imágenes, debemos evaluar el **modelo** con un conjunto de prueba, no necesariamente modificar la interfaz. Antes de reemplazarlo, define qué métrica y qué datos justificarán la nueva versión.

## 25. Cloud y Edge Computing

La aplicación desplegada mediante Streamlit Community Cloud representa una implementación **cloud**:

```text
Usuario
   ↓
Internet
   ↓
Aplicación Streamlit
   ↓
ONNX Runtime
   ↓
Modelo
   ↓
Predicción
```

Como alternativa conceptual, el mismo modelo podría ejecutarse cerca de la fuente de datos mediante **Edge Computing**, siempre que el dispositivo disponga de los recursos necesarios.

La elección entre cloud, local y Edge depende de factores como:

- conectividad;
- latencia;
- privacidad;
- recursos disponibles;
- costo;
- facilidad de actualización.


### Comparación de modalidades

| Modalidad | Lugar de ejecución | Ventaja | Restricción |
|---|---|---|---|
| Local | Computador del usuario | Puede funcionar sin Internet | Depende del equipo |
| Cloud | Servidor remoto | Acceso por URL | Requiere conectividad |
| Edge | Dispositivo cercano a la captura | Menor transferencia de datos | Recursos limitados |

**Reflexión:** ¿cómo desplegarías este clasificador sin conexión a Internet? Necesitarías una ejecución local o edge, manteniendo el contrato de inferencia.

## 26. Actividad integradora: producto final del semestre

A partir de la CNN entrenada en **NB04** y exportada a ONNX en **NB07**:

1. Recupera desde Drive el ONNX, el orden de clases y el contrato.
2. Verifica que el modelo acepte RGB 128 × 128 normalizado entre 0 y 1.
3. Prueba inferencia con una imagen de piedra, papel o tijera.
4. Genera la aplicación `app.py` con Streamlit.
5. Comprueba que muestra imagen, clase, confianza y probabilidades.
6. Genera `requirements.txt`, `README.md` y conserva los JSON.
7. Descarga y extrae el ZIP del proyecto.
8. Sube los **seis archivos** a un repositorio GitHub.
9. Despliega desde ese repositorio en Streamlit Community Cloud.
10. **Obtén y prueba la URL pública**.
11. Registra pruebas con imágenes nuevas y errores observados.
12. Explica qué cambiarías antes de publicar una nueva versión del modelo.

**Evidencia final esperada:** enlace al repositorio GitHub, **URL funcional de Streamlit** y registro de pruebas. El objetivo no se cumple solo ejecutando Colab.

## 27. Cierre del semestre

Has recorrido el ciclo de un producto de aprendizaje automático:

**NB04: entrenar una CNN → NB07: optimizar experimentalmente y exportar a ONNX → NB08: integrar en Streamlit, publicar en GitHub y desplegar en la nube.**

El resultado es una **aplicación web funcional**, capaz de recibir imágenes nuevas y clasificarlas como **piedra, papel o tijera**, con su confianza y probabilidades.

**Recuerda:** la URL pública y las pruebas funcionales deben verificarse realmente; la ejecución de este notebook prepara el despliegue, pero **no lo realiza por sí sola**.

### Reflexión final

Un producto de machine learning requiere más que entrenar: **contrato, inferencia, interfaz, distribución, pruebas, versionado y mantenimiento**.

**Preguntas:** ¿cómo distinguirías un error de interfaz de una predicción equivocada? ¿Qué cambiarías si un usuario reporta fallos frecuentes? ¿Por qué es importante conservar la versión exacta del modelo ONNX?

**Evidencia del aprendizaje:** mostrar la URL pública **y poder explicar cómo llegó allí el modelo**.